In [34]:
import pandas as pd
import numpy as np
import os
import warnings
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score,cross_validate,KFold
from sklearn.metrics import f1_score,auc,accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [46]:
X_train = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/Predicting_Heart_Disease/train.csv',index_col = 'id')
X_test = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/Predicting_Heart_Disease/test.csv',index_col = 'id')

In [47]:
X_train.info

<bound method DataFrame.info of         Age  Sex  Chest pain type   BP  Cholesterol  FBS over 120  \
id                                                                  
0        58    1                4  152          239             0   
1        52    1                1  125          325             0   
2        56    0                2  160          188             0   
3        44    0                3  134          229             0   
4        58    1                4  140          234             0   
...     ...  ...              ...  ...          ...           ...   
629995   56    0                1  110          226             0   
629996   54    1                4  128          249             1   
629997   67    1                4  130          275             0   
629998   52    1                4  140          199             0   
629999   51    0                2  130          199             0   

        EKG results  Max HR  Exercise angina  ST depression  Slope of 

In [48]:
y_train = X_train['Heart Disease'].map({
    'Presence' : 1,
    'Absence' : 0
})
X_train.drop(columns = 'Heart Disease',inplace = True)

In [49]:
print(np.unique(X_train['Chest pain type'].to_numpy()))

[1 2 3 4]


In [50]:
Lab_cols = ['Sex','Chest pain type','FBS over 120','EKG results','Exercise angina','Slope of ST','Number of vessels fluro','Thallium']
num_cols = [col for col in X_train.columns if col not in Lab_cols]

model = Pipeline(steps = [
    ('preprocess',ColumnTransformer(transformers = [('num',StandardScaler(),num_cols),
                                                    ('Lab','passthrough',Lab_cols)])),
    ('model',XGBClassifier(
        n_estimators = 1500,
        max_depth = 8,
        learning_rate = 0.01,
        reg_alpha = 0,
        reg_lambda = 1,
        subsample = 0.8,
        colsample_bytree = 0.8,
        random_state = 42,
        n_jobs = -1,
    ))
])

In [42]:
kf = KFold(n_splits = 5,shuffle = True,random_state = 42)
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv = kf,
    scoring = 'roc_auc'
)
print(scores)

[0.9550241  0.95524466 0.95581285 0.95466214 0.95492178]


In [51]:
model.fit(X_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'BP', 'Cholesterol',
                                                   'Max HR', 'ST depression']),
                                                 ('Lab', 'passthrough',
                                                  ['Sex', 'Chest pain type',
                                                   'FBS over 120',
                                                   'EKG results',
                                                   'Exercise angina',
                                                   'Slope of ST',
                                                   'Number of vessels fluro',
                                                   'Thallium'])])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsampl...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.01,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=8, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=1500, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [52]:
preds = (model.predict_proba(X_test)[:,1] >= 0.5).astype(int)
submission = pd.DataFrame({
    'id' : X_test.index,
    'Heart Disease' : preds
})
submission.to_csv('submission.csv',index = False)